[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onuralpszr/litert-llm-cookbook/blob/main/colab/17_tokenizer.ipynb)

# Example 17: Tokenizer

Use engine.tokenize and engine.detokenize to inspect how text maps to token IDs and back. Includes a token budget guard that truncates a prompt to a maximum count before sending.

In [ ]:
!pip install litert-lm-api-nightly litert-lm -q

In [ ]:
import os
import subprocess

model_url = 'https://huggingface.co/litert-community/gemma-4-E4B-it-litert-lm/resolve/main/gemma-4-E4B-it.litertlm?download=true'
output_path = '/content/gemma-4-E4B-it.litertlm'

if not os.path.exists(output_path):
    subprocess.run(['curl', '-L', model_url, '-o', output_path], check=True)
    print(f'Model downloaded to {output_path}')
else:
    print(f'Model already exists at {output_path}')

In [ ]:
import litert_lm

MODEL_PATH = "/content/gemma-4-E4B-it.litertlm"

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

TOKEN_BUDGET = 50

samples = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "LiteRT-LM runs large language models completely on-device.",
    "Tokenisation splits text into the atomic units a model actually sees.",
]

with litert_lm.Engine(MODEL_PATH) as engine:
    print("=== Part A: encoding ===")
    for text in samples:
        ids = engine.tokenize(text)
        recovered = engine.detokenize(ids)
        print(f"Text    : {text!r}")
        print(f"Token ids ({len(ids)}): {ids}")
        print(f"Decoded : {recovered!r}")
        print()

    print(f"=== Part B: budget guard (limit = {TOKEN_BUDGET} tokens) ===")
    long_prompt = " ".join(["word"] * 80)
    ids = engine.tokenize(long_prompt)
    if len(ids) > TOKEN_BUDGET:
        truncated_ids = ids[:TOKEN_BUDGET]
        truncated_text = engine.detokenize(truncated_ids)
        print(f"Prompt was {len(ids)} tokens — truncated to {len(truncated_ids)}.")
        print(f"Truncated text: {truncated_text!r}")
    else:
        print(f"Prompt fits within budget ({len(ids)} tokens).")